# Phase 1 on TIDE: finish and fix before any new experiments

Run top to bottom. Cell 1 pulls the new code (same as `tide_setup.ipynb` cells 1-2). Then:

1. **E4** (E1 base + scale/appearance augmentation) trains in the background, one seed at a time.
2. **Threshold-free comparison** of E1 / E2 / E3 from the stored sweeps (CPU, seconds).
3. **AGDD per view**: in-box scores for the 29 crack boxes plus crop montages (CPU, about a minute per run).
4. **NCUGD check** with the real models at 640 / 1024 / 1280 and a very low confidence floor (GPU, a few minutes; fine while E4 trains, the L40 has room).
5. Results table with AGDD reported per view.

Outputs land in `~/glass_crack/outputs/results/` (`compare_runs.md`, `agdd_box_scores.md`, `agdd_crack_crops_<run>.jpg`) and `~/glass_crack/outputs/ncugd_check/<run>/report.md`. Paste the tables back into the README when done.

In [ ]:
%%bash
# 1. Pull the Phase 1 code and re-run the idempotent setup
cd ~/glass_crack/cracked && git pull --ff-only && git log -1 --oneline
bash ~/glass_crack/cracked/scripts/tide/setup.sh 2>&1 | tail -15

## E4: robustness augmentation on the E1 setup
Start `E4_s0`; when `status` shows it finished, start `E4_s1`, then `E4_s2` (3 seeds are needed: seed spread is about +-0.05 recall).

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh
SEED=0   # then 1, then 2
bash "$CODE_DIR/scripts/tide/job.sh" start E4_s$SEED glass_crack_models.train.train_yolo \
    --config e4_yolo_robust.yaml --seed $SEED --device 0 --workers "$WORKERS" --compute-units "TIDE $GPU_NAME"
bash "$CODE_DIR/scripts/tide/job.sh" status

## Threshold-free comparison (no GPU)
Recall at fixed precision, FP/100 there, PR area, best F1 (oracle, for reference only), AGDD per view and NCUGD. Mean +- SD over seeds.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
ls "$OUT_ROOT/evals"
"$PY" -m glass_crack_models.eval.compare_runs --runs E1_s0 E1_s1 E1_s2 E2_s0 E2_s1 E2_s2 E3_s0

## AGDD: in-box scores per view (no GPU)
Highest score inside each crack / scratches box in `images/` vs `image/`, share of boxes reaching the frozen threshold, and how often `image/` scores lower for the same box. Montages show each crack box in both views with the score heat map.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
"$PY" -m glass_crack_models.eval.agdd_box_scores --runs E1_s0 E1_s1 E1_s2 E2_s0 E2_s1 E2_s2 E3_s0 \
    --montage E1_s0 E2_s0 E3_s0

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
res = Path.home() / 'glass_crack/outputs/results'
for run in ['E1_s0', 'E2_s0', 'E3_s0']:
    p = res / f'agdd_crack_crops_{run}.jpg'
    if p.exists():
        display(Markdown(f'**{run}**: left `images/`, right `image/` (green = labeled crack box)'), Image(str(p), width=1100))

## NCUGD: blind, or below the threshold? (GPU, a few minutes)
Runs the real checkpoints at 640 / 1024 / 1280 px, letterbox and stretch, through the same predictor as evaluation. YOLO uses a 0.001 confidence floor, so faint responses the cached maps hide become visible. A 4-class checkpoint is scored as crack or scratch.

In [ ]:
%%bash
source ~/glass_crack/cracked/scripts/tide/env.sh && cd "$CODE_DIR"
NC="$DATA_ROOT/processed/ncugd_external_test"
thr() { "$PY" -c "import json,sys; print(json.load(open(sys.argv[1]))['threshold'])" "$OUT_ROOT/evals/$1/operating_point.json"; }
check() { run=$1; shift
    echo "== $run (frozen threshold $(thr $run))"
    "$PY" scripts/check_ncugd_pipeline.py --data "$NC" --op-threshold "$(thr $run)" --device cuda:0 \
        --out "$OUT_ROOT/ncugd_check/$run" "$@" 2>&1 | tail -4; }
check E1_s0 --backend model --weights "$OUT_ROOT/runs/E1_s0/weights/best.pt"
check E2_s0 --backend model --weights "$OUT_ROOT/runs/E2_s0/weights/best.pt"
check E3_s0 --backend model --weights "$OUT_ROOT/runs/E3_s0/best.pth"

In [ ]:
%%bash
for r in E1_s0 E2_s0 E3_s0; do echo "################ $r"; head -60 ~/glass_crack/outputs/ncugd_check/$r/report.md; done

## Results table, AGDD per view
One row per run and checkpoint (re-evaluating a finished run now replaces its row instead of adding a duplicate; older duplicates disappear the next time any row is written). The averaged AGDD pair recall is deliberately left out.

In [ ]:
import os, pandas as pd
p = os.path.expanduser('~/glass_crack/outputs/results/results.csv')
df = pd.read_csv(p).drop_duplicates(['exp_id', 'seed', 'checkpoint_sha256'], keep='last')
cols = ['exp_id', 'seed', 'threshold', 'test_recall', 'test_precision', 'test_fp100', 'test_mask_iou', 'ncugd_recall',
        'agdd_images_recall', 'agdd_images_crack_recall', 'agdd_image_recall', 'agdd_image_crack_recall', 'notes']
df[[c for c in cols if c in df.columns]].sort_values(['exp_id', 'seed'])